# Exercise 07 : Approval in a Subagent

Open `exercises/exercise07-approval-in-a-subagent.ipynb` and run the setup cell. Build a supervisor whose parts subagent pauses for approval before recording an order by replacing the `____` blanks. The solution is `solutions/exercise07-approval-in-a-subagent.ipynb`.

Worked examples to revisit : module 07 demo01 · interrupts, demo02b · in-a-subagent (demo02-interrupts-in-multi-agent). ARI510 exercise13 · human-approval supplies the order and reviewer case.

Made up, not from work. Use the supplied Aurelia data and fictional policy values.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.24", "langchain-quickjs==0.3.9", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict
from uuid import UUID, uuid4

from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)

counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=650, max_retries=0)
started = time.perf_counter()
DB = Path(data("capstone/service.db")).resolve()
LIMIT = {"recursion_limit": 24}

def part_facts(part_no):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?",
                         (part_no,)).fetchone()
    if row is None:
        raise ValueError("Unknown Aurelia part")
    return dict(row)

def check(label, condition):
    print("check :", label, "PASS" if condition else "FAIL")
    assert condition, label

def finish_metrics(slug, server_usage=None):
    rows = server_usage or []
    result = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "0.3")),
              "calls": counter.calls + len(rows),
              "input_tokens": counter.input_tokens + sum(r.get("input_tokens", 0) for r in rows),
              "output_tokens": counter.output_tokens + sum(r.get("output_tokens", 0) for r in rows),
              "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", json.dumps(result))
    Path(slug + "-metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

## Step 1 : Order policy

Fill the manager threshold and quantity bounds. All requests require a person. The supplied role policy adapts ARI510 exercise13's part-order case.

In [3]:
ORDERS = {}
tool_entries = {}
review_entries = {}
REQUESTS = [
    {"order_id": "A-approve", "part_no": "MR-HF-01", "quantity": 5},
    {"order_id": "A-edit", "part_no": "CT-FN-02", "quantity": 3},
    {"order_id": "A-reject", "part_no": "CT-DM-04", "quantity": 1},
]

def reviewer(total_eur):
    return "manager" if total_eur > 10000 else "lead"

def validated_order(order):
    facts = part_facts(order["part_no"])
    quantity = order["quantity"]
    if type(quantity) is not int or not 0 < quantity <= facts["stock"]:
        raise ValueError("Quantity must be positive and within stock")
    return {**order, "total_eur": quantity * facts["price_eur"], "reviewer": reviewer(quantity * facts["price_eur"])}

for request in REQUESTS:
    print("order :", validated_order(request))
check("reviewer boundary", reviewer(10000) == "lead" and reviewer(10001) == "manager")
check("edit recomputes price", validated_order({**REQUESTS[1], "quantity": 1})["total_eur"] == 420)

order : {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550.0, 'reviewer': 'lead'}
order : {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 3, 'total_eur': 1260.0, 'reviewer': 'lead'}
order : {'order_id': 'A-reject', 'part_no': 'CT-DM-04', 'quantity': 1, 'total_eur': 12900.0, 'reviewer': 'manager'}
check : reviewer boundary PASS
check : edit recomputes price PASS


## Step 2 : Parts subagent

Fill the interrupt call and decision branches. The counter intentionally observes replay. The ledger represents a simulated action and writes only after approval. An edit may reduce quantity ; an increase needs a fresh review.

In [4]:
class Purchase(TypedDict):
    order: dict
    note: str
    result: dict

def prepare(state):
    order = validated_order(state["order"])
    note = llm.invoke("Write one short Aurelia review note from these supplied order facts only. "
                      "No order has been recorded.\n" + json.dumps(order)).content
    return {"order": order, "note": note}

def review_order(state):
    order = state["order"]
    key = order["order_id"]
    review_entries[key] = review_entries.get(key, 0) + 1
    decision = interrupt({"order": order, "note": state["note"], "options": ["approve", "edit", "reject"]})
    action = decision["action"]
    if action == "reject":
        return {"result": {"order_id": key, "status": "rejected", "reason": decision["note"]}}
    if action == "edit":
        quantity = decision["quantity"]
        if quantity > order["quantity"]:
            raise ValueError("An increased quantity needs a fresh review")
        order = validated_order({**order, "quantity": quantity})
    elif action != "approve":
        raise ValueError("Unknown approval action")
    recorded = {**order, "status": "recorded", "decision": action, "external_action": False}
    if key in ORDERS and ORDERS[key] != recorded:
        raise ValueError("Order id already recorded with different values")
    ORDERS.setdefault(key, recorded)
    return {"result": recorded}

b = StateGraph(Purchase)
b.add_node("prepare", prepare)
b.add_node("review", review_order)
b.add_edge(START, "prepare")
b.add_edge("prepare", "review")
b.add_edge("review", END)
parts_subagent = b.compile()
check("child has no separate saver", parts_subagent.checkpointer is None)

check : child has no separate saver PASS


## Step 3 : Supervisor tool

Complete the nested invocation and parent saver. Keep the tool result as JSON. Preserve the propagated interrupt instead of returning it as an ordinary tool result.

In [5]:
@tool
def parts_specialist(order_id: str, part_no: str, quantity: int) -> str:
    """Ask the parts subagent to review and record one simulated Aurelia order."""
    tool_entries[order_id] = tool_entries.get(order_id, 0) + 1
    result = parts_subagent.invoke({"order": {"order_id": order_id, "part_no": part_no, "quantity": quantity}})
    return json.dumps(result["result"])

supervisor = create_agent(llm, tools=[parts_specialist], checkpointer=InMemorySaver(),
    system_prompt="Call parts_specialist exactly once with the requested order id, part and quantity. "
                  "After its result, report the recorded or rejected status and stop. "
                  "An edited result is authoritative. Never retry a rejected order.")
check("supervisor saver installed", supervisor.checkpointer is not None)
check("order ledger empty", ORDERS == {})

check : supervisor saver installed PASS
check : order ledger empty PASS


## Step 4 : Top-level pause

Run the requests and read the top-level payloads. Record the two entry counters before supplying a decision.

In [6]:
configs = {}
paused = {}
for request in REQUESTS:
    key = request["order_id"]
    cfg = {"configurable": {"thread_id": str(uuid4())}, "recursion_limit": 24}
    configs[key] = cfg
    out = supervisor.invoke({"messages": [HumanMessage("Request this Aurelia order: " + json.dumps(request))]},
                            cfg, version="v2")
    paused[key] = out
    print("top-level interrupts :", [i.value for i in out.interrupts])
    print("top-level next :", supervisor.get_state(cfg).next)
    check("one nested approval", len(out.interrupts) == 1)
    check("requested order preserved", out.interrupts[0].value["order"]["order_id"] == key
          and out.interrupts[0].value["order"]["part_no"] == request["part_no"]
          and out.interrupts[0].value["order"]["quantity"] == request["quantity"])
    check("supervisor paused in tools", supervisor.get_state(cfg).next == ("tools",))
check("nothing recorded before approval", not ORDERS)
check("first entries counted", all(tool_entries[r["order_id"]] == review_entries[r["order_id"]] == 1 for r in REQUESTS))

top-level interrupts : [{'order': {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550.0, 'reviewer': 'lead'}, 'note': 'No order has been recorded.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS


top-level interrupts : [{'order': {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 3, 'total_eur': 1260.0, 'reviewer': 'lead'}, 'note': 'Review note: No order has been recorded.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS


top-level interrupts : [{'order': {'order_id': 'A-reject', 'part_no': 'CT-DM-04', 'quantity': 1, 'total_eur': 12900.0, 'reviewer': 'manager'}, 'note': 'No order has been recorded.', 'options': ['approve', 'edit', 'reject']}]
top-level next : ('tools',)
check : one nested approval PASS
check : requested order preserved PASS
check : supervisor paused in tools PASS
check : nothing recorded before approval PASS
check : first entries counted PASS


## Step 5 : Decisions

Fill the decisions and resume the saved supervisor configs. Compare the simulated ledger with both counters. The completed preparation node, including its model call, stays checkpointed ; the paused tool wrapper and review node execute again.

In [7]:
decisions = {
    "A-approve": {"action": "approve"},
    "A-edit": {"action": "edit", "quantity": 1},
    "A-reject": {"action": "reject", "note": "Repair the detector first. Manager declines the purchase."},
}
completed = {}
for key, decision in decisions.items():
    out = supervisor.invoke(Command(resume=decision), configs[key], version="v2")
    completed[key] = out
    print("decision :", key, decision)
    for message in out.value["messages"]:
        if isinstance(message, ToolMessage):
            print("parts result :", message.content)
    print("supervisor :", out.value["messages"][-1].content)
    check("completed without another interrupt", not out.interrupts)
check("approved quantity retained", ORDERS["A-approve"]["quantity"] == 5 and ORDERS["A-approve"]["total_eur"] == 1550)
check("edited quantity and price", ORDERS["A-edit"]["quantity"] == 1 and ORDERS["A-edit"]["total_eur"] == 420)
rejected_results = [json.loads(m.content) for m in completed["A-reject"].value["messages"] if isinstance(m, ToolMessage)]
check("rejection recorded without order", "A-reject" not in ORDERS and rejected_results[-1]["status"] == "rejected")
print("entry counts :", {"tool": tool_entries, "review": review_entries})
print("simulated orders :", ORDERS)
check("code before interrupt replayed", all(tool_entries[r["order_id"]] == review_entries[r["order_id"]] == 2 for r in REQUESTS))
check("each approved action recorded once", len(ORDERS) == 2 and all(not o["external_action"] for o in ORDERS.values()))

decision : A-approve {'action': 'approve'}
parts result : {"order_id": "A-approve", "part_no": "MR-HF-01", "quantity": 5, "total_eur": 1550.0, "reviewer": "lead", "status": "recorded", "decision": "approve", "external_action": false}
supervisor : Order A-approve was recorded.
check : completed without another interrupt PASS


decision : A-edit {'action': 'edit', 'quantity': 1}
parts result : {"order_id": "A-edit", "part_no": "CT-FN-02", "quantity": 1, "total_eur": 420.0, "reviewer": "lead", "status": "recorded", "decision": "edit", "external_action": false}
supervisor : Recorded: order A-edit was edited to quantity 1 (total €420.00).
check : completed without another interrupt PASS


decision : A-reject {'action': 'reject', 'note': 'Repair the detector first. Manager declines the purchase.'}
parts result : {"order_id": "A-reject", "status": "rejected", "reason": "Repair the detector first. Manager declines the purchase."}
supervisor : Order A-reject was rejected. Repair the detector first; the manager declined the purchase.
check : completed without another interrupt PASS
check : approved quantity retained PASS
check : edited quantity and price PASS
check : rejection recorded without order PASS
entry counts : {'tool': {'A-approve': 2, 'A-edit': 2, 'A-reject': 2}, 'review': {'A-approve': 2, 'A-edit': 2, 'A-reject': 2}}
simulated orders : {'A-approve': {'order_id': 'A-approve', 'part_no': 'MR-HF-01', 'quantity': 5, 'total_eur': 1550.0, 'reviewer': 'lead', 'status': 'recorded', 'decision': 'approve', 'external_action': False}, 'A-edit': {'order_id': 'A-edit', 'part_no': 'CT-FN-02', 'quantity': 1, 'total_eur': 420.0, 'reviewer': 'lead', 'status': 'recorded', 'decision'

### If time permits

- Try editing to quantity 0 or above stock and inspect the validation failure
- Print the supervisor state history and locate the pending approval checkpoint
- Add a second manager review after the lead approves an expensive order

In [8]:
finish_metrics("exercise07-approval-in-a-subagent")

metrics : {"model": "openai/gpt-6-luna", "rps": 0.06, "calls": 9, "input_tokens": 1336, "output_tokens": 192, "seconds": 151.8}
